# Stage 1 — 10: native-resolution forensic group-CV

A7 resizes the full frame before the V-JEPA crop. This notebook intentionally
uses the repository's other Stage 1 path:

`native-resolution frame -> native 256×256 patch -> forensic model`

No frame is resized before patch extraction.

The default shortlist uses three complementary models already implemented in
this repository:

- `bayar_resnet18`: constrained residual / manipulation traces,
- `frequency`: periodic / moiré spectrum cues,
- `lcdf`: chromaticity + FMAG-inspired frequency stream.

The notebook reuses the same group folds as 09B, produces true OOF predictions,
and averages fold models on the untouched fixed DLC val. Notebook 11 consumes
those OOF and fixed-val files.


## 1. Setup


In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

import warnings

# ============================================================
# Suppress known harmless third-party warnings
# ============================================================
warnings.filterwarnings(
    "ignore",
    message=r".*torch\.backends\.cuda\.sdp_kernel.*deprecated.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*Importing from timm\.models\.layers is deprecated.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*The parameter 'pretrained' is deprecated.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*Arguments other than a weight enum or .* for 'weights'.*",
    category=UserWarning,
)

REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")
    if not (REPO_ROOT / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "--single-branch", REPO_URL, str(REPO_ROOT),
        ], check=True)
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if current_branch != BRANCH:
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if dirty:
            print("WARNING: local repo has changes; git pull skipped.")
        else:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError("Run this notebook inside Blackbox-Detection repository.")

os.chdir(REPO_ROOT)

# Keep Colab's binary scientific stack intact. Install only the Stage 1 extras
# and then this repository editable with --no-deps, matching the current notebooks.
COLAB_EXTRAS = [
    "av>=15,<17", "timm==1.0.15", "fvcore==0.1.5.post20221221",
    "iopath==0.1.10", "yacs==0.1.8", "einops==0.8.1",
    "omegaconf==2.3.0", "hydra-core==1.3.2", "easydict==1.13",
]
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS,
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)
], check=True)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.utils import load_checkpoint, seed_everything

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required = {"DLC_ROOT": DLC_ROOT, "DLC_SPLIT_CSV": DLC_SPLIT_CSV}
    missing = [f"{k}: {v}" for k, v in required.items() if not v.exists()]
    if missing:
        raise FileNotFoundError("Missing required Drive paths:\n  " + "\n  ".join(missing))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("repo   :", REPO_ROOT)
print("branch :", BRANCH)
print("commit :", GIT_COMMIT)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from blackbox_detection.stage1.dataset import (
    Stage1ForensicDataset, build_dataloader, forensic_batch_adapter,
)
from blackbox_detection.stage1.evaluator import (
    AggregationConfig, Stage1Evaluator, evaluate_predictions, save_predictions,
)
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_forensic_samplers
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig
from blackbox_detection.stage1.transforms import PatchAugmentConfig, build_forensic_transforms

FORENSIC_CFG = yaml.safe_load((CONFIG_DIR / "forensic.yaml").read_text(encoding="utf-8"))
RUN_ROOT = OUTPUT_ROOT / "10_forensic_groupcv"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_NAMES = ["bayar_resnet18", "frequency", "lcdf"]
N_FOLDS = 5
FORCE_RETRAIN = False
NUM_WORKERS = 2
print("models:", MODEL_NAMES)
print("output:", RUN_ROOT)


## 2. DLC manifests and fold assignments


In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}

def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")

def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    index = {}
    counts = {}
    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"
        if not clips_root.is_dir():
            raise FileNotFoundError(f"DLC clips directory not found: {clips_root}")
        count = 0
        for path in clips_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            rel = path.relative_to(clips_root).with_suffix("").as_posix()
            key = (source, _normalize_rel_text(rel))
            if key in index and index[key] != str(path):
                raise ValueError(f"Duplicate DLC video key: {key}")
            index[key] = str(path)
            count += 1
        counts[source] = count
    print("indexed DLC videos:", counts)
    return index

DLC_VIDEO_INDEX = _build_dlc_video_index()

def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)
    key = (source, clip_id)
    if key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[key]
    matches = []
    for (src, rel), path in DLC_VIDEO_INDEX.items():
        if src != source:
            continue
        if rel.startswith(clip_id + "/") or rel.endswith("/" + clip_id) or rel == clip_id:
            matches.append(path)
    if len(matches) == 1:
        return matches[0]
    leaf = Path(clip_id).name
    matches = [
        path for (src, rel), path in DLC_VIDEO_INDEX.items()
        if src == source and Path(rel).name == leaf
    ]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"Cannot resolve DLC source={source!r}, clip_id={clip_id!r}")

def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()
    required = {
        "clip_id", "class", "source", "document_type", "document_id",
        "group", "split", "device", "condition",
    }
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"dlc_split.csv missing columns: {missing}")
    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["source"] = raw["source"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()
    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No DLC rows for split={split_name!r}")
    frame["label"] = frame["class"].map({"original": "ORIGINAL", "rerecorded": "RERECORDED"})
    if frame["label"].isna().any():
        raise ValueError("Unexpected DLC class value found.")
    frame["video_id"] = "dlc__" + frame["clip_id"].astype(str).str.replace("/", "__", regex=False)
    frame["dataset"] = "dlc2021"
    frame["scene_type"] = "document"
    frame["video_path"] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame["source"], frame["clip_id"])
    ]
    keep = [
        "video_path", "label", "video_id", "dataset", "scene_type",
        "clip_id", "source", "document_type", "document_id", "group",
        "device", "condition",
    ]
    out = frame[keep].reset_index(drop=True)
    missing_paths = [p for p in out["video_path"] if not Path(p).is_file()]
    if missing_paths:
        raise FileNotFoundError(
            f"{split_name}: {len(missing_paths)} missing videos; examples={missing_paths[:3]}"
        )
    return out


In [ ]:
train_df = load_dlc_manifest("train")
fixed_val_df = load_dlc_manifest("val")
fold_file = OUTPUT_ROOT / "09b_grouped_probe_cv" / "fold_assignments.csv"
if fold_file.is_file():
    assignments = pd.read_csv(fold_file)
    train_df = train_df.merge(
        assignments[["video_id", "fold"]], on="video_id", how="left", validate="one_to_one"
    )
    if train_df["fold"].isna().any():
        raise RuntimeError("09B fold assignments do not cover all DLC train videos.")
    train_df["fold"] = train_df["fold"].astype(int)
    if train_df["fold"].nunique() != N_FOLDS:
        raise ValueError(
            f"09B fold file has {train_df['fold'].nunique()} folds, expected {N_FOLDS}."
        )
    print("reused folds:", fold_file)
else:
    print("09B fold file not found; regenerating the same StratifiedGroupKFold.")
    y = train_df["label"].map({"ORIGINAL":0, "RERECORDED":1}).to_numpy()
    groups = train_df["group"].astype(str).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_id = np.full(len(train_df), -1, dtype=np.int64)
    for fold, (_, va_idx) in enumerate(splitter.split(train_df, y, groups)):
        fold_id[va_idx] = fold
    train_df["fold"] = fold_id

for fold in range(N_FOLDS):
    tr = train_df.loc[train_df["fold"].ne(fold)]
    va = train_df.loc[train_df["fold"].eq(fold)]
    overlap = set(tr["group"].astype(str)) & set(va["group"].astype(str))
    if overlap:
        raise RuntimeError(f"fold {fold}: group leakage detected")
    if va["label"].nunique() != 2:
        raise RuntimeError(f"fold {fold}: validation has one class only")
    print(f"fold {fold}: train={len(tr)} val={len(va)} labels={va['label'].value_counts().to_dict()}")


## 3. Config helpers


In [ ]:
def merge_config(defaults: dict, overrides: dict) -> dict:
    merged = copy.deepcopy(defaults)
    for section, values in overrides.items():
        if isinstance(values, dict):
            merged.setdefault(section, {})
            merged[section] = {**merged[section], **copy.deepcopy(values)}
        else:
            merged[section] = copy.deepcopy(values)
    return merged

def build_model_config(model_name: str) -> dict:
    if model_name not in FORENSIC_CFG["models"]:
        raise ValueError(f"Unknown forensic model: {model_name}")
    return merge_config(FORENSIC_CFG["defaults"], FORENSIC_CFG["models"][model_name])

ADAPTER = forensic_batch_adapter()


## 4. Train grouped forensic models and collect OOF/fixed-val predictions


In [ ]:
all_model_summaries = {}

for model_name in MODEL_NAMES:
    print("\n" + "#" * 88)
    print("MODEL:", model_name)
    print("#" * 88)
    cfg = build_model_config(model_name)
    model_root = RUN_ROOT / model_name
    model_root.mkdir(parents=True, exist_ok=True)
    data_cfg = cfg["data"]
    aug_cfg = cfg["augmentation"]
    train_cfg = cfg["train"]
    patch_size = int(cfg["model"]["params"]["patch_size"])

    patch_augment = PatchAugmentConfig(
        hflip_prob=float(aug_cfg["hflip_prob"]),
        vflip_prob=float(aug_cfg["vflip_prob"]),
        rot90_prob=float(aug_cfg["rot90_prob"]),
        spectral_augment_prob=float(aug_cfg["spectral_augment_prob"]),
        spectral_augment_alpha_range=tuple(aug_cfg["spectral_augment_alpha_range"]),
        spectral_augment_beta_std=float(aug_cfg["spectral_augment_beta_std"]),
        spectral_augment_keep_outside=bool(aug_cfg["spectral_augment_keep_outside"]),
    )
    train_transform, val_transform = build_forensic_transforms(train_config=patch_augment)
    train_frame_sampler, train_patch_sampler = build_forensic_samplers(
        train=True,
        num_frames=int(data_cfg["train_num_frames"]),
        num_patches=int(data_cfg["train_num_patches"]),
        patch_size=patch_size,
    )
    val_frame_sampler, val_patch_sampler = build_forensic_samplers(
        train=False,
        num_frames=int(data_cfg["val_num_frames"]),
        num_patches=int(data_cfg["val_num_patches"]),
        patch_size=patch_size,
        val_grid=int(data_cfg["val_patch_grid"]),
    )

    def make_forensic_loader(frame: pd.DataFrame, *, train: bool, seed: int):
        dataset = Stage1ForensicDataset(
            frame.reset_index(drop=True),
            frame_sampler=train_frame_sampler if train else val_frame_sampler,
            patch_sampler=train_patch_sampler if train else val_patch_sampler,
            transform=train_transform if train else val_transform,
            patch_size=patch_size,
            on_error="zero",
            deterministic=not train,
        )
        return build_dataloader(
            dataset,
            batch_size=int(data_cfg["batch_size"] if train else data_cfg["val_batch_size"]),
            shuffle=train,
            num_workers=NUM_WORKERS,
            seed=seed,
            drop_last=False,
            persistent_workers=NUM_WORKERS > 0,
        )

    oof_frames = []
    fixed_frames = []
    fold_summaries = []

    for fold in range(N_FOLDS):
        print("\n---", model_name, "fold", fold, "---")
        fold_dir = model_root / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        tr_df = train_df.loc[train_df["fold"].ne(fold)].drop(columns=["fold"]).reset_index(drop=True)
        va_df = train_df.loc[train_df["fold"].eq(fold)].drop(columns=["fold"]).reset_index(drop=True)
        seed = int(train_cfg["seed"]) + fold
        train_loader = make_forensic_loader(tr_df, train=True, seed=seed)
        fold_val_loader = make_forensic_loader(va_df, train=False, seed=seed)
        fixed_val_loader = make_forensic_loader(fixed_val_df, train=False, seed=int(train_cfg["seed"]))

        seed_everything(seed, deterministic=False)
        model = build_stage1_model(
            model_name,
            finetune_mode=cfg["model"]["finetune_mode"],
            unfreeze_last_n=int(cfg["model"]["unfreeze_last_n"]),
            **cfg["model"]["params"],
        )
        print("parameters:", count_parameters(model))
        best_ckpt = fold_dir / "best.pt"
        if FORCE_RETRAIN or not best_ckpt.is_file():
            trainer_cfg = TrainConfig(
                epochs=int(train_cfg["epochs"]),
                learning_rate=float(train_cfg["learning_rate"]),
                head_learning_rate=(
                    float(train_cfg["head_learning_rate"])
                    if train_cfg.get("head_learning_rate") is not None else None
                ),
                weight_decay=float(train_cfg["weight_decay"]),
                warmup_ratio=float(train_cfg["warmup_ratio"]),
                grad_accum_steps=int(train_cfg["grad_accum_steps"]),
                max_grad_norm=float(train_cfg["max_grad_norm"]),
                amp=bool(train_cfg["amp"]),
                label_smoothing=float(train_cfg.get("label_smoothing", 0.0)),
                early_stopping_patience=int(train_cfg["early_stopping_patience"]),
                eval_every=int(train_cfg["eval_every"]),
                seed=seed,
                output_dir=fold_dir,
                model_name=f"{model_name}_10_fold{fold}",
                wandb_enabled=False,
            )
            trainer = Stage1Trainer(
                model, trainer_cfg, adapter=ADAPTER,
                aggregation=AggregationConfig(frame_method="mean", video_method="mean"),
                model_config={
                    "name": model_name,
                    "params": cfg["model"]["params"],
                    "experiment": "10_forensic_groupcv",
                    "fold": fold,
                },
            )
            outcome = trainer.fit(train_loader, fold_val_loader)
            print(
                f"best epoch={outcome.best_epoch} "
                f"F1={outcome.best_macro_f1:.4f} thr={outcome.best_threshold:.4f}"
            )
            del trainer, outcome

        load_checkpoint(best_ckpt, model=model, map_location="cpu", restore_rng_state=False)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device).eval()
        evaluator = Stage1Evaluator(
            model, ADAPTER, device=device, amp=False,
            aggregation=AggregationConfig(frame_method="mean", video_method="mean"),
        )
        oof_result = evaluator.evaluate(fold_val_loader, threshold=0.5, search_threshold=False)
        oof_pred = oof_result.predictions.copy(); oof_pred["fold"] = fold
        oof_frames.append(oof_pred)
        fixed_result = evaluator.evaluate(fixed_val_loader, threshold=0.5, search_threshold=False)
        fixed_pred = fixed_result.predictions.copy(); fixed_pred["fold"] = fold
        fixed_frames.append(fixed_pred)
        fold_summary = {
            "fold": fold,
            "oof_macro_f1_at_0.5": float(oof_result.macro_f1_at_default),
            "fixed_val_macro_f1_at_0.5": float(fixed_result.macro_f1_at_default),
        }
        fold_summaries.append(fold_summary)
        save_predictions(oof_pred, fold_dir / "oof_predictions.csv")
        save_predictions(fixed_pred, fold_dir / "fixed_val_predictions.csv")
        del model, evaluator, train_loader, fold_val_loader, fixed_val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof = pd.concat(oof_frames, ignore_index=True)
    if oof["video_id"].duplicated().any():
        raise RuntimeError(f"{model_name}: duplicate OOF video ids")
    if set(oof["video_id"]) != set(train_df["video_id"]):
        raise RuntimeError(f"{model_name}: OOF coverage mismatch")
    oof_result = evaluate_predictions(oof, threshold=0.5, search_threshold=False)
    save_predictions(oof_result.predictions, model_root / "oof_predictions.csv")

    stacked = pd.concat(fixed_frames, ignore_index=True)
    ensemble = (
        stacked.groupby(["video_id", "label", "dataset"], as_index=False)
        .agg(prob_rerecorded=("prob_rerecorded", "mean"), fold_std=("prob_rerecorded", "std"))
    )
    ensemble["fold_std"] = ensemble["fold_std"].fillna(0.0)
    ensemble["prob_original"] = 1.0 - ensemble["prob_rerecorded"]
    fixed_ensemble_result = evaluate_predictions(ensemble, threshold=0.5, search_threshold=False)
    save_predictions(fixed_ensemble_result.predictions, model_root / "val_predictions.csv")

    model_summary = {
        "model": model_name,
        "oof_macro_f1_at_0.5": float(oof_result.macro_f1_at_default),
        "fixed_val_ensemble_macro_f1_at_0.5": float(fixed_ensemble_result.macro_f1_at_default),
        "mean_fixed_val_fold_std": float(ensemble["fold_std"].mean()),
        "folds": fold_summaries,
    }
    all_model_summaries[model_name] = model_summary
    (model_root / "summary.json").write_text(
        json.dumps(model_summary, indent=2), encoding="utf-8"
    )

summary_table = pd.DataFrame(all_model_summaries.values()).sort_values(
    ["oof_macro_f1_at_0.5", "fixed_val_ensemble_macro_f1_at_0.5"],
    ascending=False, kind="mergesort",
)
summary_table.to_csv(RUN_ROOT / "model_summary.csv", index=False)
display(summary_table)


## 5. OOF error-overlap / hard-fold / worst-group diagnostics

The forensic branch is not selected by standalone Macro-F1 alone. Compare every
forensic OOF prediction against the same 483-video 09B V-JEPA grouped OOF
predictions.

The main signals are:

- how many V-JEPA OOF errors the forensic model corrects,
- how many new errors it introduces where V-JEPA was already correct,
- how many errors remain common to both branches,
- performance and error overlap on the automatically detected hardest V-JEPA fold,
- which V-JEPA weak groups are actually corrected by the forensic branch.

The oracle score below is only a complementarity ceiling using ground truth; it
is not a deployable fusion score and must not be used as a submission result.


In [ ]:
from blackbox_detection.utils.metrics import stage1_score

BASE_OOF_PATH = OUTPUT_ROOT / "09b_grouped_probe_cv" / "oof_predictions.csv"
BASE_FOLD_PATH = OUTPUT_ROOT / "09b_grouped_probe_cv" / "fold_assignments.csv"

if not BASE_OOF_PATH.is_file():
    raise FileNotFoundError(
        f"09B grouped OOF predictions are required before diagnostics: {BASE_OOF_PATH}"
    )
if not BASE_FOLD_PATH.is_file():
    raise FileNotFoundError(
        f"09B fold assignments are required before diagnostics: {BASE_FOLD_PATH}"
    )


def _load_oof_predictions(path: Path, prefix: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {
        "video_id", "label", "dataset", "prob_rerecorded", "prediction"
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")
    return frame[
        ["video_id", "label", "dataset", "prob_rerecorded", "prediction"]
    ].rename(
        columns={
            "prob_rerecorded": f"p_{prefix}",
            "prediction": f"pred_{prefix}",
        }
    )


def _macro_f1_if_binary(group: pd.DataFrame, pred_col: str) -> float:
    if len(group) == 0 or group["label"].nunique() < 2:
        return float("nan")
    return float(
        stage1_score(
            group["label"].astype(str).tolist(),
            group[pred_col].astype(str).tolist(),
        )
    )


base = _load_oof_predictions(BASE_OOF_PATH, "vjepa")
fold_assignments = pd.read_csv(BASE_FOLD_PATH)[["video_id", "fold"]].copy()

meta_cols = [
    "video_id", "group", "document_id", "document_type", "device", "condition",
]
meta = train_df[meta_cols].drop_duplicates("video_id").copy()

base = (
    base
    .merge(fold_assignments, on="video_id", how="left", validate="one_to_one")
    .merge(meta, on="video_id", how="left", validate="one_to_one")
)
if base["fold"].isna().any():
    raise RuntimeError("09B OOF contains rows without a fold assignment.")
base["fold"] = base["fold"].astype(int)
base["vjepa_correct"] = base["pred_vjepa"].eq(base["label"])

# ------------------------------------------------------------
# 09B baseline fold diagnostics and automatic hard-fold choice
# ------------------------------------------------------------
base_fold_rows = []
for fold, g in base.groupby("fold", sort=True):
    base_fold_rows.append(
        {
            "fold": int(fold),
            "num_videos": int(len(g)),
            "macro_f1_at_0.5": _macro_f1_if_binary(g, "pred_vjepa"),
            "num_errors": int((~g["vjepa_correct"]).sum()),
        }
    )

base_fold_table = pd.DataFrame(base_fold_rows).sort_values(
    ["macro_f1_at_0.5", "num_errors"],
    ascending=[True, False],
    na_position="last",
    kind="mergesort",
).reset_index(drop=True)

valid_fold_rows = base_fold_table.loc[base_fold_table["macro_f1_at_0.5"].notna()]
if len(valid_fold_rows):
    HARD_FOLD = int(valid_fold_rows.iloc[0]["fold"])
else:
    HARD_FOLD = int(base_fold_table.iloc[0]["fold"])

print("=== 09B V-JEPA OOF FOLDS ===")
display(base_fold_table)
print("auto-selected hard fold:", HARD_FOLD)

# ------------------------------------------------------------
# 09B baseline group diagnostics
# ------------------------------------------------------------
base_group_rows = []
for group_name, g in base.groupby("group", sort=True):
    base_group_rows.append(
        {
            "group": str(group_name),
            "num_videos": int(len(g)),
            "base_errors": int((~g["vjepa_correct"]).sum()),
            "base_macro_f1": _macro_f1_if_binary(g, "pred_vjepa"),
            "base_mean_margin": float((g["p_vjepa"] - 0.5).abs().mean()),
        }
    )

base_group_table = pd.DataFrame(base_group_rows).sort_values(
    ["base_errors", "base_macro_f1", "base_mean_margin"],
    ascending=[False, True, True],
    na_position="last",
    kind="mergesort",
).reset_index(drop=True)

print("\n=== 09B V-JEPA WORST GROUPS ===")
display(base_group_table.head(20))

# ------------------------------------------------------------
# V-JEPA vs each forensic model: error complementarity
# ------------------------------------------------------------
complementarity_rows = []
fold_overlap_rows = []
all_group_overlap = {}
all_error_cases = {}

for model_name in MODEL_NAMES:
    forensic_path = RUN_ROOT / model_name / "oof_predictions.csv"
    if not forensic_path.is_file():
        print(f"skip diagnostics for {model_name}: missing {forensic_path}")
        continue

    forensic = _load_oof_predictions(forensic_path, "forensic")
    wide = base.merge(
        forensic,
        on=["video_id", "label", "dataset"],
        how="inner",
        validate="one_to_one",
    )
    if len(wide) != len(base):
        raise RuntimeError(
            f"{model_name}: OOF coverage mismatch {len(wide)} != {len(base)}"
        )

    wide["forensic_correct"] = wide["pred_forensic"].eq(wide["label"])
    wide["base_error"] = ~wide["vjepa_correct"]
    wide["forensic_error"] = ~wide["forensic_correct"]
    wide["corrected_by_forensic"] = wide["base_error"] & wide["forensic_correct"]
    wide["introduced_by_forensic"] = wide["vjepa_correct"] & wide["forensic_error"]
    wide["common_error"] = wide["base_error"] & wide["forensic_error"]
    wide["prediction_disagreement"] = wide["pred_vjepa"].ne(wide["pred_forensic"])

    base_errors = int(wide["base_error"].sum())
    forensic_errors = int(wide["forensic_error"].sum())
    corrected = int(wide["corrected_by_forensic"].sum())
    introduced = int(wide["introduced_by_forensic"].sum())
    common = int(wide["common_error"].sum())
    union_errors = int((wide["base_error"] | wide["forensic_error"]).sum())

    # Ground-truth oracle ceiling: use V-JEPA when correct; otherwise use forensic.
    # This measures whether complementary information exists. It is not deployable.
    oracle_pred = np.where(
        wide["vjepa_correct"],
        wide["pred_vjepa"],
        wide["pred_forensic"],
    )
    oracle_f1 = float(
        stage1_score(wide["label"].astype(str).tolist(), oracle_pred.tolist())
    )

    complementarity_rows.append(
        {
            "forensic_model": model_name,
            "vjepa_oof_macro_f1": _macro_f1_if_binary(wide, "pred_vjepa"),
            "forensic_oof_macro_f1": _macro_f1_if_binary(wide, "pred_forensic"),
            "vjepa_errors": base_errors,
            "forensic_errors": forensic_errors,
            "corrected_vjepa_errors": corrected,
            "vjepa_error_correction_rate": (
                float(corrected / base_errors) if base_errors else 0.0
            ),
            "introduced_errors": introduced,
            "common_errors": common,
            "error_union": union_errors,
            "error_jaccard": (
                float(common / union_errors) if union_errors else 0.0
            ),
            "prediction_disagreements": int(wide["prediction_disagreement"].sum()),
            "oracle_macro_f1": oracle_f1,
            "hard_fold": HARD_FOLD,
        }
    )

    # Fold-level complementarity. This is where fold 0 should emerge naturally
    # if it remains the hardest 09B V-JEPA fold.
    for fold, g in wide.groupby("fold", sort=True):
        fold_overlap_rows.append(
            {
                "forensic_model": model_name,
                "fold": int(fold),
                "num_videos": int(len(g)),
                "vjepa_macro_f1": _macro_f1_if_binary(g, "pred_vjepa"),
                "forensic_macro_f1": _macro_f1_if_binary(g, "pred_forensic"),
                "vjepa_errors": int(g["base_error"].sum()),
                "forensic_errors": int(g["forensic_error"].sum()),
                "corrected_vjepa_errors": int(g["corrected_by_forensic"].sum()),
                "introduced_errors": int(g["introduced_by_forensic"].sum()),
                "common_errors": int(g["common_error"].sum()),
                "is_hard_fold": bool(int(fold) == HARD_FOLD),
            }
        )

    # Group-level complementarity: specifically track the groups where V-JEPA
    # already showed OOF failures in 09B.
    group_rows = []
    for group_name, g in wide.groupby("group", sort=True):
        group_rows.append(
            {
                "group": str(group_name),
                "num_videos": int(len(g)),
                "vjepa_errors": int(g["base_error"].sum()),
                "forensic_errors": int(g["forensic_error"].sum()),
                "corrected_vjepa_errors": int(g["corrected_by_forensic"].sum()),
                "introduced_errors": int(g["introduced_by_forensic"].sum()),
                "common_errors": int(g["common_error"].sum()),
                "vjepa_macro_f1": _macro_f1_if_binary(g, "pred_vjepa"),
                "forensic_macro_f1": _macro_f1_if_binary(g, "pred_forensic"),
                "vjepa_mean_margin": float((g["p_vjepa"] - 0.5).abs().mean()),
                "forensic_mean_margin": float((g["p_forensic"] - 0.5).abs().mean()),
            }
        )

    group_table = pd.DataFrame(group_rows).sort_values(
        [
            "vjepa_errors",
            "corrected_vjepa_errors",
            "common_errors",
            "forensic_errors",
        ],
        ascending=[False, False, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    all_group_overlap[model_name] = group_table

    error_cases = wide.loc[
        wide["base_error"] | wide["forensic_error"],
        [
            "video_id", "label", "fold", "group", "document_type", "device",
            "condition", "p_vjepa", "pred_vjepa", "p_forensic",
            "pred_forensic", "base_error", "forensic_error",
            "corrected_by_forensic", "introduced_by_forensic", "common_error",
        ],
    ].sort_values(
        ["base_error", "corrected_by_forensic", "common_error", "group"],
        ascending=[False, False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)
    all_error_cases[model_name] = error_cases

    group_table.to_csv(
        RUN_ROOT / model_name / "oof_group_complementarity.csv",
        index=False,
    )
    error_cases.to_csv(
        RUN_ROOT / model_name / "oof_error_cases_vs_vjepa.csv",
        index=False,
    )

if not complementarity_rows:
    raise RuntimeError(
        "No forensic OOF predictions were available for complementarity diagnostics."
    )

complementarity_summary = pd.DataFrame(complementarity_rows).sort_values(
    [
        "corrected_vjepa_errors",
        "common_errors",
        "introduced_errors",
        "forensic_oof_macro_f1",
    ],
    ascending=[False, True, True, False],
    kind="mergesort",
).reset_index(drop=True)

fold_complementarity = pd.DataFrame(fold_overlap_rows).sort_values(
    ["forensic_model", "fold"],
    kind="mergesort",
).reset_index(drop=True)

complementarity_summary.to_csv(
    RUN_ROOT / "oof_complementarity_summary.csv",
    index=False,
)
fold_complementarity.to_csv(
    RUN_ROOT / "oof_fold_complementarity.csv",
    index=False,
)

print("\n=== FORENSIC COMPLEMENTARITY SUMMARY ===")
display(complementarity_summary)

print(f"\n=== AUTO-SELECTED HARD FOLD {HARD_FOLD} ===")
display(
    fold_complementarity.loc[
        fold_complementarity["fold"].eq(HARD_FOLD)
    ].sort_values(
        ["corrected_vjepa_errors", "common_errors", "forensic_errors"],
        ascending=[False, True, True],
        kind="mergesort",
    )
)

for model_name, group_table in all_group_overlap.items():
    print(f"\n=== {model_name}: V-JEPA WEAK GROUPS / FORENSIC CORRECTIONS ===")
    display(group_table.loc[group_table["vjepa_errors"].gt(0)].head(30))

    print(f"\n=== {model_name}: ERROR CASES VS V-JEPA ===")
    display(all_error_cases[model_name].head(30))


## How to read the diagnostics

Prefer a forensic branch that **corrects V-JEPA errors without merely trading
them for new errors**. In particular, inspect:

1. `corrected_vjepa_errors` and `vjepa_error_correction_rate`,
2. `common_errors` — lower means more complementary failure modes,
3. `introduced_errors`,
4. the automatically selected hard fold (09B currently makes this fold 0),
5. the V-JEPA weak groups that produced OOF errors.

Standalone forensic Macro-F1 is secondary. A weaker standalone model can still
be useful for notebook 11 if its errors are sufficiently different from V-JEPA.


## 6. Save run summary


In [ ]:
payload = {
    "experiment": "10_forensic_groupcv",
    "git_commit": GIT_COMMIT,
    "real_dlc_only": True,
    "native_resolution_before_crop": True,
    "n_folds": N_FOLDS,
    "models": all_model_summaries,
    "hard_fold": int(HARD_FOLD),
    "oof_complementarity": complementarity_summary.to_dict(orient="records"),
}
(RUN_ROOT / "summary.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print("saved to:", RUN_ROOT)


## Notes

This notebook deliberately does not mix CCD or synthetic recapture into the
forensic training data. Its purpose is to test whether **real DLC,
native-resolution forensic cues** provide prediction diversity that complements
V-JEPA.
